In [ ]:
import sys
sys.path.append("..")
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from model.depth_anything_v2.dpt_align_lora import DepthAnythingV2_lora
from model.depth_anything_v2.dpt_align import DepthAnythingV2
from model.epde_modal import EPDE
from model.epde.utils import clean_pretrained_weight

import os
import matplotlib
import matplotlib.pyplot as plt
import numpy as np

from da2_metric_depth.dataset.eventscape import EventScape
from da2_metric_depth.dataset.eventscape_voxel import EventScape_voxel

from da2_metric_depth.dataset.mvsec import MVSEC
from da2_metric_depth.dataset.mvsec_voxel import MVSEC_voxel
from dataset.eventscape import EventScape as EventScape_mixed
from dataset.mvsec import MVSEC as MVSEC_mixed
from torch.utils.data import DataLoader

In [2]:
def get_variance_maps(features, use_cls_token=True):
    feature_maps = []
    if use_cls_token:
        for feature in features:
            fea = feature[0]
            feature_maps.append(fea)
    else:
        feature_maps = features

    var_maps = []
    for fea in feature_maps:
        fea = fea.permute(0, 2, 1)
        var = torch.var(fea, dim=1, keepdim=True)
        var = var.permute(0, 2, 1)
        var_maps.append(var)

    return var_maps


def token2feature(tokens, patch_grid_size):
    """add token transfer to feature"""
    B, L, D = tokens.shape
    # H = W = int(L**0.5)
    W, H = patch_grid_size[0], patch_grid_size[1]
    x = tokens.permute(0, 2, 1).view(B, D, H, W).contiguous()
    return x


model_configs = {
    'vits': {'encoder': 'vits', 'features': 64, 'out_channels': [48, 96, 192, 384]},
    'vitb': {'encoder': 'vitb', 'features': 128, 'out_channels': [96, 192, 384, 768]},
    'vitl': {'encoder': 'vitl', 'features': 256, 'out_channels': [256, 512, 1024, 1024]},
    'vitg': {'encoder': 'vitg', 'features': 384, 'out_channels': [1536, 1536, 1536, 1536]}
}

In [17]:
encoder = "vitl"
max_depth = 1
size = (1036, 1036)
# mvsec, mvsec_voxel, eventscape, eventscape_voxel, eventscape_mixed, mvsec_mixed
dataset = "eventscape_mixed"
load_from = "/data/coding/code/da2-prompt-tuning/exp/fuse_log_l1_eventscape_fuse_cor_20250114_110446/latest.pth"
# load_from = "/data/coding/upload-data/checkpoints/depth_anything_v2_vitl.pth"
outdir = "/data/coding/code/da2-prompt-tuning/vis_feature_maps/fuse_logl1_fea_eventscape_event_1036_2"

In [ ]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
# DEVICE = 'cpu'

# depth_anything = DepthAnythingV2(**{**model_configs[encoder], 'return_feature': True})
# depth_anything = DepthAnythingV2_lora(**{**model_configs[encoder], 'return_feature': True})
depth_anything = EPDE(model_name=encoder, max_depth=max_depth, event_voxel_chans=3, return_feature=True)

checkpoint = torch.load(load_from, map_location='cpu')
checkpoint = clean_pretrained_weight(checkpoint)

depth_anything.load_state_dict(checkpoint)
print(f"Model weights load from {load_from} successfully!")
depth_anything = depth_anything.to(DEVICE).eval()

In [18]:
if dataset == "eventscape":
    valset = EventScape(
        "/data/coding/code/da2-prompt-tuning/dataset/splits/eventscape/vis_feature.txt",
        "val",
        normalized_d=False,
        size=size,
    )
elif dataset == "mvsec":
    valset = MVSEC(
        "/data/coding/code/da2-prompt-tuning/dataset/splits/mvsec/vis_feature.txt",
        "val",
        normalized_d=False,
        size=size,
    )
elif dataset == "mvsec_voxel":
    valset = MVSEC_voxel(
        "/data/coding/code/da2-prompt-tuning/dataset/splits/mvsec/vis_feature.txt",
        "val",
        normalized_d=False,
        size=size,
    )
elif dataset == "mvsec_mixed":
    valset = MVSEC_mixed(
        "/data/coding/code/da2-prompt-tuning/dataset/splits/mvsec/vis_feature.txt",
        "val",
        normalized_d=False,
        size=size,
    )
elif dataset == "eventscape_mixed":
    valset = EventScape_mixed(
        "/data/coding/code/da2-prompt-tuning/dataset/splits/eventscape/vis_feature.txt",
        "val",
        normalized_d=False,
        size=size,
    )
else:
    valset = EventScape_voxel(
        "/data/coding/code/da2-prompt-tuning/dataset/splits/eventscape/vis_feature.txt",
        "val",
        normalized_d=False,
        size=size,
    )

valloader = DataLoader(
        valset,
        batch_size=1,
        pin_memory=True,
        num_workers=4,
        drop_last=True,
    )

In [19]:
os.makedirs(outdir, exist_ok=True)
npy_dir = os.path.join(outdir, 'npy')
vis_dir = os.path.join(outdir, 'vis')
fea_dir = os.path.join(outdir, 'fea')
os.makedirs(npy_dir, exist_ok=True)
os.makedirs(vis_dir, exist_ok=True)
os.makedirs(fea_dir, exist_ok=True)

cmap = matplotlib.colormaps.get_cmap('Spectral')

In [ ]:
depth_anything.eval()
for k, sample in enumerate(valloader):
    filename = sample["image_path"][0]
    print(f'Progress {k+1}/{len(valloader)}: {filename}')
    
    raw_image = cv2.imread(filename)
    h, w = raw_image.shape[0], raw_image.shape[1]
    
    with torch.no_grad():
        if "mixed" in dataset:
            img = sample["input"].to(DEVICE)
        else:
            img = sample["image"].to(DEVICE)
        
        depth, var_maps = depth_anything(img)
        depth = F.interpolate(
            depth[:, None], (h, w), mode="bilinear", align_corners=True
        )[0, 0]
    
    depth = depth.cpu().numpy()

    # Save predicted depth map
    output_path = os.path.join(npy_dir, os.path.splitext(os.path.basename(filename))[0] + '.npy')
    np.save(output_path, depth)
    
    # vis depth_map
    depth = (depth - depth.min()) / (depth.max() - depth.min()) * 255.0
    depth = depth.astype(np.uint8)
    depth = (cmap(depth)[:, :, :3] * 255)[:, :, ::-1].astype(np.uint8)
    
    output_path = os.path.join(vis_dir, os.path.splitext(os.path.basename(filename))[0] + '.png')
    # Merge depth map and raw rgb
    split_region = np.ones((raw_image.shape[0], 50, 3), dtype=np.uint8) * 255
    combined_result = cv2.hconcat([raw_image, split_region, depth])
    cv2.imwrite(output_path, combined_result)
    
    # vis feature map
    for i, var in enumerate(var_maps):
        output_path = os.path.join(fea_dir, os.path.splitext(os.path.basename(filename))[0] + f'_{i}.png')
        var = var.cpu().numpy().squeeze()
        # var = (cmap(var)[:, :, :3] * 255)[:, :, ::-1].astype(np.uint8)
        # cv2.imwrite(output_path, var)
        np.save(output_path, var)

In [ ]:
# Single Image
import os
import matplotlib
import numpy as np

filename = "/data/coding/upload-data/data/DSEC/000767.png"
outdir = "/data/coding/upload-data/data/DSEC"
cmap = matplotlib.colormaps.get_cmap('Spectral')

raw_image = cv2.imread(filename)
depth, var_maps = depth_anything.infer_image(raw_image, 1036)

depth = (depth - depth.min()) / (depth.max() - depth.min()) * 255.0
depth = depth.astype(np.uint8)
depth = (cmap(depth)[:, :, :3] * 255)[:, :, ::-1].astype(np.uint8)

cv2.imwrite(os.path.join(outdir, os.path.splitext(os.path.basename(filename))[0] + '_depth.png'), depth)
# vis feature map
for i, var in enumerate(var_maps):
    output_path = os.path.join(outdir, os.path.splitext(os.path.basename(filename))[0] + f'fea_{i}.npy')
    var = var.cpu().numpy().squeeze()
    # var = (cmap(var)[:, :, :3] * 255)[:, :, ::-1].astype(np.uint8)
    # cv2.imwrite(output_path, var)
    np.save(output_path, var)

#### Vis feature Map

In [21]:
def vis_depth_map(dep, show_colorbar=False, cmap_name="Spectral", save_fig=False, save_path="test.png"):
    cmap = matplotlib.colormaps.get_cmap(cmap_name)

    fig, ax = plt.subplots()
    fig.subplots_adjust(left=0, right=1, top=1, bottom=0)
    plt.axis("off")

    im = ax.imshow(dep, cmap=cmap, extent=[0, dep.shape[1], 0, dep.shape[0]])
    if show_colorbar:
        plt.colorbar(im)

    dpi = fig.get_dpi()
    fig.set_size_inches(dep.shape[1] / dpi, dep.shape[0] / dpi)

    if save_fig:
        plt.savefig(save_path)
    plt.show()

In [ ]:
p = "/data/coding/code/da2-prompt-tuning/vis_feature_maps/align_eventscape_event_1036_1_da2/npy/05_084_0124_image.npy"
d = np.load(p)
# d = 1.0 / d
print(d.min(), d.max())
vis_depth_map(d, show_colorbar=True)

In [22]:
# EventScape
# patch_size = (38, 19) # 266
# patch_size = (50, 25) # 350
# patch_size = (74, 37) # 518 
patch_size = (148, 74) # 1036

# MVSEC
# patch_size = (25, 19) # 266
# patch_size = (49, 37) # 518
# patch_size = (98, 74) # 1036

# DSEC
# patch_size = (97, 74)# 1036
# patch_size = (99, 74)# 1036
# patch_size = (74, 99)# 1036
# patch_size = (103, 77)# 1036

In [ ]:
# dir = "/data/coding/code/da2-prompt-tuning/vis_feature_maps/align_log_l1_eventscape_event_1036_1_lora"
dir = outdir

fea_npy_dir = os.path.join(dir, "fea")
fea_vis_dir = os.path.join(dir, "fea_var_vis")
os.makedirs(fea_vis_dir, exist_ok=True)

fea_npy_paths = os.listdir(fea_npy_dir)
for p in fea_npy_paths:
    npy_path = os.path.join(fea_npy_dir, p)
    fea = torch.from_numpy(np.load(npy_path))
    fea = fea.unsqueeze(0)
    fea = token2feature(fea, patch_size)
    var = torch.var(fea, dim=1, keepdim=True).squeeze().numpy()
    
    file_name = os.path.basename(p).replace(".npy", "")
    # file_name = p.split('_')[0] + '_' + p.split('_')[1][0] + ".png"
    out_path = os.path.join(fea_vis_dir, file_name)
    vis_depth_map(var, cmap_name="viridis", save_fig=True, save_path=out_path)

In [ ]:
fea_path = "/data/coding/upload-data/data/DSEC/000040fea_0.npy"
fea = torch.from_numpy(np.load(fea_path))
fea = fea.unsqueeze(0)
print(fea.shape)
fea = token2feature(fea, patch_size)
print(fea.shape)

var = torch.var(fea, dim=1, keepdim=True).squeeze().numpy()
vis_depth_map(var, cmap_name="viridis", save_fig=False)

Mean = torch.mean(fea, dim=1, keepdim=True).squeeze().numpy()
vis_depth_map(Mean, cmap_name="viridis", save_fig=False)

Max = torch.max(fea, dim=1, keepdim=True)[0].squeeze().numpy()
vis_depth_map(Max, cmap_name="viridis", save_fig=False)

Min = torch.min(fea, dim=1, keepdim=True)[0].squeeze().numpy()
vis_depth_map(Min, cmap_name="viridis", save_fig=False)

In [ ]:
dir = "/home/sph/event/da2-prompt-tuning/vis_feature_maps/da2_vitl_mvsec_fea_1036"

dep_npy_dir = os.path.join(dir, "npy")
dep_vis_dir = os.path.join(dir, "dep_vis")
os.makedirs(dep_vis_dir, exist_ok=True)

dep_npy_paths = os.listdir(dep_npy_dir)
for p in dep_npy_paths:
    npy_path = os.path.join(dep_npy_dir, p)
    dep = np.load(npy_path)
    
    file_name = p.replace("npy", "png")
    out_path = os.path.join(dep_vis_dir, file_name)
    vis_depth_map(dep, save_fig=True, save_path=out_path)